In [0]:
%sql
use catalog end_to_end_projects;
use schema sports_equipments_sales;

In [0]:
df_silver = spark.table("sports_equipment_sales_silver")
display(df_silver.limit(5))
print(f"Count of Records in Silver Delta Table: {df_silver.count()}")
df_silver.printSchema()

# Incremental Dimension Tables

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

## Incremental Gold Data Modeling : SCD Type 1 MERGE

In [0]:
df_silver = spark.table("sports_equipment_sales_silver")
display(df_silver.limit(5))
print(f"Updated Silver Count: {df_silver.count()}")

In [0]:
df_customer_incremental = df_silver.select("customer_id", "customer_name", "customer_city", "customer_state", "last_updated")\
    .withColumn("row_num", row_number().over(Window.partitionBy("customer_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")
display(df_customer_incremental.limit(5))
print(f"Count of Incremental Customer Records: {df_customer_incremental.count()}")

> Step 46: SCD Type 1 MERGE into Existing dim_customers Table

In [0]:
dim_customers = spark.table("dim_customers")
print(f"Count of Dim Customers: {dim_customers.count()}")

In [0]:
df_new_customers = df_customer_incremental.join(dim_customers, on = "customer_id", how = "left_anti")
print(f"Count of New Customers: {df_new_customers.count()}")

In [0]:
max_customer_sk = spark.sql("""
                            select max(customer_sk)
                            from dim_customers
                            """
                            )
max_customer_sk = max_customer_sk.first()[0]
print(f"Max Customer SK: {max_customer_sk}")                            

In [0]:
df_new_customers = df_new_customers.withColumn("customer_sk", row_number().over(Window.orderBy("customer_id")) + max_customer_sk)
display(df_new_customers.limit(5))
print(f"Count of records: {df_new_customers.count()}")

In [0]:
df_new_customers.printSchema()

In [0]:
df_customer_merge = df_customer_incremental.alias("src").join(df_new_customers.alias("new"), on = "customer_id", how = "left")\
    .select(col("customer_id"), col("src.customer_name"), col("src.customer_city"), col("src.customer_state"), col("new.customer_sk"))
print(f"Count of Records: {df_customer_merge.count()}")

## SCD Type 1 MERGE into Existing dim_customers Table

In [0]:
df_customer_merge.createOrReplaceTempView("customer_merge_source")

In [0]:
%sql
merge into dim_customers as target
using customer_merge_source as source
on target.customer_id = source.customer_id

when matched then
update set
target.customer_name = source.customer_name,
target.customer_city = source.customer_city,
target.customer_state = source.customer_state

when not matched then
insert(customer_id, customer_name, customer_city, customer_state, customer_sk)
values(source.customer_id, source.customer_name, source.customer_city, source.customer_state, source.customer_sk)


In [0]:
dim_customers = spark.table("dim_customers")
print(f"Count of Dim Customers Records After Merge: {dim_customers.count()}")
display(dim_customers.limit(20))

In [0]:
%sql
select * from dim_customers where customer_sk >2000;

In [0]:
print(f"Unqiue Customer ID Count: {dim_customers.select("customer_id").distinct().count()}")
print(f"Unique Customer SK Count: {dim_customers.select("customer_sk").distinct().count()}")
print(f"Null Customer SK Count: {dim_customers.filter(col("customer_sk").isNull()).count()}")

`Dim Products`

In [0]:
df_products_incremental = df_silver.select("product_id", "product_name", "product_category", "product_brand", "last_updated")\
    .withColumn("row_num", row_number().over(Window.partitionBy("product_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")
display(df_products_incremental.limit(5))
print(f"Count of Incremental Product Records: {df_products_incremental.count()}")

In [0]:
# Read existing Product Dimension and verify the Product count
dim_products = spark.table("dim_products")
print(f"Count of Existing Products: {dim_products.count()}")

In [0]:
# Identify New Products
dim_new_products = df_products_incremental.join(dim_products, on = "product_id", how = "left_anti")
print(f"Count Of New Products: {dim_new_products.count()}")

In [0]:
# Find The Maximum SK
max_product_sk = spark.sql("""
                            select max(product_sk)
                            from dim_products
                            """
)
max_product_sk = max_product_sk.first()[0]
print(f"Max Product SK: {max_product_sk}")

In [0]:
# Generate Surrogate Keys For New Products
dim_new_products = dim_new_products.withColumn("product_sk", row_number().over(Window.orderBy("product_id")) + max_product_sk)
display(dim_new_products.limit(5))
print(f"Count of Product records: {dim_new_products.count()}")

In [0]:
# Prepare Product MERGE Source
dim_product_merge = df_products_incremental.alias("src").join(dim_new_products.alias("new"), on = "product_id", how = "left")\
    .select(col("product_id"), col("src.product_name"), col("src.product_category"), col("src.product_brand"), col("new.product_sk"))
print(f"Count Of Records: {dim_product_merge.count()}")

In [0]:
dim_product_merge.createOrReplaceTempView("product_merge_Source")

In [0]:
%sql
merge into dim_products as target
using product_merge_source as source
on target.product_id = source.product_id

when matched then 
update set
target.product_name = source.product_name,
target.product_category = source.product_category,
target.product_brand = source.product_brand

when not matched then 
insert(product_id, product_name, product_category, product_brand, product_sk)
values(source.product_id, source.product_name, source.product_category, source.product_brand, source.product_sk)

In [0]:
dim_products = spark.table("dim_products")
print(f"Count of Records after merge: {dim_products.count()}")

In [0]:
print(f"Unqiue Product ID Count: {dim_products.select("product_id").distinct().count()}")
print(f"Unique Product SK Count: {dim_products.select("product_sk").distinct().count()}")
print(f"Null Product SK Count: {dim_products.filter(col("product_sk").isNull()).count()}")


> ## dim_incremental_store

In [0]:
dim_incremental_store = df_silver.select("store_id", "store_name","store_city", "store_region", "last_updated")\
    .withColumn("row_num", row_number().over(Window.partitionBy("store_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")
print(f"Count Of Incremental Store Records: {dim_incremental_store.count()}")
display(dim_incremental_store.limit(5))


In [0]:
#Read Existing Store Dimension and Verify The count
dim_store = spark.table("dim_store")
print(f"Count Of Existing Dim Store Records: {dim_store.count()}")
display(dim_store.limit(5))

In [0]:
#Identify the New Store Records
dim_new_stores = dim_incremental_store.join(dim_store, on = "store_id", how = "left_anti")
print(f"Count Of New Stores : {dim_new_stores.count()}")

In [0]:
max_store_sk = spark.sql("""
                         select max(store_sk)
                         from dim_store
                         """
)
max_store_sk = max_store_sk.first()[0]
print(f"Max Store SK: {max_store_sk}")
                         

In [0]:
dim_new_stores = dim_new_stores.withColumn("store_sk", row_number().over(Window.orderBy("store_id")) + max_store_sk)
display(dim_new_stores.limit(5))

In [0]:
dim_store_merge = dim_incremental_store.alias("src").join(dim_new_stores.alias("new"), on = "store_id", how = "left")\
    .select(col("store_id"), col("src.store_name"), col("src.store_city"), col("src.store_region"), col("new.store_sk"))
print(f"Count of Records: {dim_store_merge.count()}")

In [0]:
dim_store_merge.createOrReplaceTempView("store_merge_source")

In [0]:
%sql
merge into dim_store as target
using store_merge_source as source
on target.store_id = source.store_id

when matched then
update set
target.store_name = source.store_name,
target.store_city = source.store_city,
target.store_region = source.store_region

when not matched then
insert(store_id, store_name, store_city, store_region, store_sk)
values(source.store_id, source.store_name, source.store_city, source.store_region, source.store_sk);

In [0]:
dim_store = spark.table("dim_store")
print(f"Count Of Records After Merge: {dim_store.count()}")

In [0]:
print(f"Unqiue Store ID Count: {dim_store.select("store_id").distinct().count()}")
print(f"Unique Store SK Count: {dim_store.select("store_sk").distinct().count()}")
print(f"Null Store SK Count: {dim_store.filter(col("store_sk").isNull()).count()}")

> ## dim_salesperson

In [0]:
dim_incremental_salesperson = df_silver.select("salesperson_id", "salesperson_name", "last_updated")\
    .withColumn("row_num", row_number().over(Window.partitionBy("salesperson_id").orderBy(col("last_updated").desc())))\
        .filter(col("row_num") == 1).drop("row_num", "last_updated")
print(f"Count of Incremental Salesperson Record:{dim_incremental_salesperson.count()}")
display(dim_incremental_salesperson.limit(5))

In [0]:
dim_salesperson = spark.table("dim_salesperson")
print(f"Count of Existing SalesPerson Records: {dim_salesperson.count()}")

In [0]:
dim_new_salesperson = dim_incremental_salesperson.join(dim_salesperson, on = "salesperson_id", how = "left_anti")
print(f"Count of New Salesperson Records: {dim_new_salesperson.count()}")

In [0]:
max_salesperson_sk = spark.sql("""
                               select max(salesperson_sk)
                               from dim_salesperson
                               """
)
max_salesperson_sk = max_salesperson_sk.first()[0]
print(f"Max Salesperson SK: {max_salesperson_sk}")

In [0]:
dim_new_salesperson = dim_new_salesperson.withColumn("salesperson_sk", row_number().over(Window.orderBy("salesperson_id")) + max_salesperson_sk)
display(dim_new_salesperson.limit(5))

In [0]:
dim_salesperson_merge = dim_incremental_salesperson.alias("src").join(dim_new_salesperson.alias("new"), on = "salesperson_id", how = "left")\
    .select(col("salesperson_id"), col("src.salesperson_name"), col("new.salesperson_sk"))
print(f"Count Of Records: {dim_salesperson_merge.count()}")

> Merge

In [0]:
dim_salesperson_merge.createOrReplaceTempView("salesperson_merge_source")

In [0]:
%sql
merge into dim_salesperson as target
using salesperson_merge_source as source
on target.salesperson_id = source.salesperson_id

when matched then 
update set
target.salesperson_name = source.salesperson_name

when not matched then 
insert(salesperson_id, salesperson_name, salesperson_sk)
values(source.salesperson_id, source.salesperson_name, source.salesperson_sk)

In [0]:
print(f"Count Of Sales Person Records After Merge: {dim_salesperson.count()}")
print(f"Count of Unique Sales Person ID: {dim_salesperson.select("salesperson_id").distinct().count()}")
print(f"Count of Unique Sales Person SK: {dim_salesperson.select("salesperson_sk").distinct().count()}")   
print(f"Count of Null sales person sks: {dim_salesperson.filter(col("salesperson_sk").isNull()).count()}")

# Incremental Fact Table

- Read the updated Silver Table and all four updated Dimension Tables.

In [0]:
df_silver = spark.table("sports_equipment_sales_silver")
print(f"Count Of records: {df_silver.count()}")
display(df_silver.limit(5))

dim_customers = spark.table("dim_customers")
dim_products = spark.table("dim_products")
dim_store = spark.table("dim_store")
dim_salesperson = spark.table("dim_salesperson")
print(f"Count Of customer records: {dim_customers.count()}")
print(f"Count Of products records: {dim_products.count()}")
print(f"Count Of store records: {dim_store.count()}")
print(f"Count Of salesperson records: {dim_salesperson.count()}")
display(dim_customers.limit(5))
display(dim_products.limit(5))
display(dim_store.limit(5))
display(dim_salesperson.limit(5))

In [0]:
df_fact_incremental = df_silver.join(dim_customers, on="customer_id", how="left")\
    .join(dim_products, on = "product_id", how = "left")\
        .join(dim_store, on = "store_id", how = "left")\
            .join(dim_salesperson, on = "salesperson_id", how = "left")
print(f"Fact Join Incremental Count: {df_fact_incremental.count()}")
df_fact_incremental.printSchema()

In [0]:
# Original Fact columns
fact_columns = [
    "sale_id", "customer_sk", "product_sk", "store_sk",
    "salesperson_sk", "sale_date", "quantity", "unit_price",
    "discount_amount", "tax_amount", "final_amount",
    "payment_method", "sale_status", "last_updated"
]

# Exclude Dimension columns and ingestion metadata
dimension_columns = set(
    dim_customers.columns +
    dim_products.columns +
    dim_store.columns +
    dim_salesperson.columns
)

# Detect additional columns dynamically from Silver
new_columns = [
    column_name for column_name in df_silver.columns
    if column_name not in fact_columns
    and column_name not in dimension_columns
    and column_name != "ingestion_timestamp"
]

# Prepare Fact MERGE source
df_fact_incremental = df_fact_incremental.select(
    *(fact_columns + new_columns)
)

print(f"Additional Fact Columns: {new_columns}")
print(f"Fact Record Count: {df_fact_incremental.count()}")
print(f"Count of Records: {df_fact_incremental.count()}")
display(df_fact_incremental.limit(5))

In [0]:
print(f"Null Customer SK Count: {df_fact_incremental.filter(col("customer_sk").isNull()).count()}")
print(f"Null Product SK Count: {df_fact_incremental.filter(col("product_sk").isNull()).count()}")
print(f"Null Store SK Count: {df_fact_incremental.filter(col("store_sk").isNull()).count()}")
print(f"Null Salesperson SK Count: {df_fact_incremental.filter(col("salesperson_sk").isNull()).count()}")

In [0]:
df_fact_incremental.createOrReplaceTempView("fact_incremental_merge")

In [0]:
%sql
merge with schema evolution into fact_sales as target
using fact_incremental_merge as source
on target.sale_id = source.sale_id

when matched then
update set *

when not matched then 
insert *;

In [0]:
fact_sales = spark.table("fact_sales")
print(f"Count of Records in Fact Sales after merge: {fact_sales.count()}")
fact_sales.printSchema()
print(f"Unqiue Sale ID Count: {fact_sales.select(col("sale_id")).distinct().count()}")